# Five-layer architecture: interface examples

This notebook is the executable usage guide for the modular architecture. The raw and standard data layers are implemented; later sections define the intended public interfaces.

## 1. Raw data layer

`RawDataManager` reads source metadata from a catalog, resolves paths under `data/`, and reports availability without interpreting source-specific columns. `source_id` is a stable dataset key, not a dynamically generated Python variable.

In [1]:
from pathlib import Path

from IPython.display import display

from src.raw import RawDataManager
from src.standard import StandardDataManager, time_bounds

project_root = Path.cwd()
data_root = project_root / "data"
catalog_path = project_root / "config/raw_data_sources.csv"

In [8]:
raw_data = RawDataManager(catalog_path, data_root)
display(raw_data.catalog)

,source_id,domain,provider,acquisition_method,file_format,local_path,source_url,download_url,api_url,remote_file_name,version,checksum_algorithm,expected_checksum,required,description,download_instructions,options_json
source_id,,,,,,,,,,,,,,,,,
osm_china_pbf,osm_china_pbf,network,Geofabrik/direct OSM export,direct,pbf,osm/china-latest.osm.pbf,https://download.geofabrik.de/asia/china.html,https://download.geofabrik.de/asia/china-lates...,,,latest,,,True,Complete China OpenStreetMap extract used to d...,Automatic: prepare downloads the current china...,{}
gem_integrated_power,gem_integrated_power,generation,Global Energy Monitor,manual,xlsx,Global-Integrated-Power-March-2026-II.xlsx,https://globalenergymonitor.org/projects/globa...,,,,March 2026,,,True,Unit- and phase-level global power project rec...,"Manual: open the GEM page, select Download dat...",{}
doe_storage,doe_storage,storage,Sandia National Laboratories,manual,json,doe_global_energy_storage_database_2022.json,https://gesdb.sandia.gov/projects.html,,,,2022,,,False,Project-level global energy storage database e...,"Manual: open the Sandia GESDB project page, ac...",{}
provincial_hourly_load,provincial_hourly_load,load,Figshare,figshare_api,xlsx,china_provincial_hourly_load_2015_2024.xlsx,https://doi.org/10.6084/m9.figshare.29832701,,https://api.figshare.com/v2/articles/29832701,Data output.xlsx,2015-2024,md5,bf730a545bc297b20d6a7ff93571992a,True,China provincial synthetic hourly load workbook.,Automatic: prepare resolves Data output.xlsx t...,{}
worldpop_population,worldpop_population,population,WorldPop,direct,tif,chn_ppp_2020_1km_aggregated_unadj.tif,https://data.worldpop.org/GIS/Population/Globa...,https://data.worldpop.org/GIS/Population/Globa...,,,2020,,,True,UN-adjusted one-kilometre population raster fo...,Automatic: prepare downloads the GeoTIFF direc...,{}
era5_china_2024,era5_china_2024,weather,Copernicus Climate Data Store,atlite_cds,nc,capacity-factors/era5_china_2024.nc,https://cds.climate.copernicus.eu/datasets/rea...,,,,2024,,,False,ERA5 China cutout containing weather fields re...,"API: create a CDS account, accept the ERA5 lic...","{""year"":2024,""x"":[73.0,136.0],""y"":[18.0,54.0],..."
province_boundaries,province_boundaries,administrative_boundary,DataV,direct,geojson,china_provinces_datav_original.geojson,https://geo.datav.aliyun.com/areas_v3/bound/10...,https://geo.datav.aliyun.com/areas_v3/bound/10...,,,current,,,True,Original province boundary GeoJSON before geom...,Automatic: prepare downloads the original prov...,{}
pypsa_technology_costs,pypsa_technology_costs,technology_cost,PyPSA technology-data,direct,csv,pypsa_technology_data_costs_2025.csv,https://github.com/PyPSA/technology-data,https://raw.githubusercontent.com/PyPSA/techno...,,,2025,,,True,Upstream technology-data cost and efficiency a...,Automatic: prepare downloads outputs/costs_202...,{}
cec_yearbook,cec_yearbook,official_statistics,China Electricity Council,manual,pdf,cec_china_electric_power_statistics_yearbook_2...,,,,,2024,,,False,Authorized China Electric Power Statistics Yea...,Manual: obtain the authorized 2024 yearbook PD...,{}


In [12]:
all_sources = raw_data.check()
one_source = raw_data.check("osm_china_pbf")
selected_sources = raw_data.check([
    "gem_integrated_power", "provincial_hourly_load",
])
display(all_sources)
display(one_source)
display(selected_sources)

,domain,provider,acquisition_method,file_format,local_path,required,exists,size_bytes,checksum,status,download_instructions
source_id,,,,,,,,,,,
osm_china_pbf,network,Geofabrik/direct OSM export,direct,pbf,/Users/jiang/Documents/Codex/nature-plan-paper...,True,True,1560217727,,available,Automatic: prepare downloads the current china...
gem_integrated_power,generation,Global Energy Monitor,manual,xlsx,/Users/jiang/Documents/Codex/nature-plan-paper...,True,True,23337385,,available,"Manual: open the GEM page, select Download dat..."
doe_storage,storage,Sandia National Laboratories,manual,json,/Users/jiang/Documents/Codex/nature-plan-paper...,False,True,7436775,,available,"Manual: open the Sandia GESDB project page, ac..."
provincial_hourly_load,load,Figshare,figshare_api,xlsx,/Users/jiang/Documents/Codex/nature-plan-paper...,True,True,41867984,bf730a545bc297b20d6a7ff93571992a,available,Automatic: prepare resolves Data output.xlsx t...
worldpop_population,population,WorldPop,direct,tif,/Users/jiang/Documents/Codex/nature-plan-paper...,True,True,49445858,,available,Automatic: prepare downloads the GeoTIFF direc...
era5_china_2024,weather,Copernicus Climate Data Store,atlite_cds,nc,/Users/jiang/Documents/Codex/nature-plan-paper...,False,False,<NA>,,credentials_required,"API: create a CDS account, accept the ERA5 lic..."
province_boundaries,administrative_boundary,DataV,direct,geojson,/Users/jiang/Documents/Codex/nature-plan-paper...,True,True,582522,,available,Automatic: prepare downloads the original prov...
pypsa_technology_costs,technology_cost,PyPSA technology-data,direct,csv,/Users/jiang/Documents/Codex/nature-plan-paper...,True,True,281386,,available,Automatic: prepare downloads outputs/costs_202...
cec_yearbook,official_statistics,China Electricity Council,manual,pdf,/Users/jiang/Documents/Codex/nature-plan-paper...,False,True,3693366,,available,Manual: obtain the authorized 2024 yearbook PD...


,domain,provider,acquisition_method,file_format,local_path,required,exists,size_bytes,checksum,status,download_instructions
source_id,,,,,,,,,,,
osm_china_pbf,network,Geofabrik/direct OSM export,direct,pbf,/Users/jiang/Documents/Codex/nature-plan-paper...,True,True,1560217727,,available,Automatic: prepare downloads the current china...


,domain,provider,acquisition_method,file_format,local_path,required,exists,size_bytes,checksum,status,download_instructions
source_id,,,,,,,,,,,
gem_integrated_power,generation,Global Energy Monitor,manual,xlsx,/Users/jiang/Documents/Codex/nature-plan-paper...,True,True,23337385,,available,"Manual: open the GEM page, select Download dat..."
provincial_hourly_load,load,Figshare,figshare_api,xlsx,/Users/jiang/Documents/Codex/nature-plan-paper...,True,True,41867984,bf730a545bc297b20d6a7ff93571992a,available,Automatic: prepare resolves Data output.xlsx t...


`prepare()` accepts the same scopes as `check()`: `raw_data.prepare("osm_china_pbf")`, `raw_data.prepare(["provincial_hourly_load", "worldpop_population"])`, or `raw_data.prepare()` for the full catalog. Direct and Figshare sources download automatically. ERA5 uses CDS credentials and atlite. GEM, Sandia storage, and the CEC yearbook return `manual_action_required` with exact instructions. Existing files are not overwritten unless `overwrite=True` is passed. The next layer obtains a path with `raw_data.get_file("osm_china_pbf")`.

In [20]:
all_data = raw_data.prepare(source_ids=["province_boundaries", "pypsa_technology_costs"], overwrite=True)

,domain,provider,acquisition_method,file_format,local_path,required,exists,size_bytes,checksum,status,download_instructions,action,error
source_id,,,,,,,,,,,,,
province_boundaries,administrative_boundary,DataV,direct,geojson,/Users/jiang/Documents/Codex/nature-plan-paper...,True,True,582522,,available,Automatic: prepare downloads the original prov...,downloaded,<NA>
pypsa_technology_costs,technology_cost,PyPSA technology-data,direct,csv,/Users/jiang/Documents/Codex/nature-plan-paper...,True,True,281386,,available,Automatic: prepare downloads outputs/costs_202...,downloaded,<NA>


## 2. Standard data layer

`StandardDataManager` exposes the stable dataset IDs `spatial`, `network`, `generation`, `storage`, `parameter`, `load`, `population`, and `resource`. Source-specific processing is controlled by `config/standard_data.toml`; changing source dependencies does not change these public IDs.

In [ ]:
standard_data = StandardDataManager(
    project_root / "config/standard_data.toml",
    raw_data=raw_data,
)
display(standard_data.check())

In [ ]:
# build() regenerates a dataset; load() reads an existing standard artifact.
generation = standard_data.load("generation")
network = standard_data.load("network")

display(generation[[
    "uid", "type", "technology", "status", "voltage_kv", "geometry"
]].head())
display(network.nodes.head())
display(network.branches.head())

In [ ]:
load = standard_data.load("load")
population = standard_data.load("population")

display(load)
display(population)
time_bounds("2024-03")

## 3. Spatiotemporal mapping layer (planned interface)

Input: canonical assets, spatial units, and time coordinates. Output: explicit mapping tables such as `generation_node_mapping` and `load_cell_node_mapping`, including method, distance, confidence, and review flags. Typical usage will be `mappings = SpatiotemporalMapping(standard_data, mapping_config).build()`.

## 4. System case layer (planned interface)

Input: canonical data, mappings, and a scenario configuration. Output: one validated `PowerSystemCase` containing static asset tables and aligned time-series arrays. Typical usage will be `case = SystemCaseBuilder(standard_data, mappings).build(case_config)`.

## 5. Application layer (planned interface)

Input: `PowerSystemCase` plus formulation and solver options. Output: standardized model results without mutating the case. Typical usage will be `result = UnitCommitmentApplication(application_config).solve(case)`; OPF and planning applications will consume the same case contract.